# ZK-SNARK Private DAO Voting System
### End-to-End Google Colab Development, Deployment and Benchmarking Pipeline
**Author:** Pragma Mumbai ZK Workshop  
**Architecture:** Groth16 (BN128) + Circom 2.0+ + Polygon Mumbai/Amoy Testnet  

Pipeline Stages:
1. Environment Setup & GPU Check (Tesla T4 / A100 / RTX 4060)
2. ZK Circuit Compilation & Trusted Setup Ceremony
3. Proof Generation & Bilinear Pairing Verification
4. Smart Contract Deployment on Polygon Mumbai/Amoy
5. Frontend Tunneling (ngrok)
6. Real Empirical Benchmarking (Proof Time, Memory, Verification)

## Section 1: Environment and GPU Setup

In [ ]:
# Check GPU Availability in Colab
import torch
import tensorflow as tf

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Running in CPU mode. In Colab, switch runtime: Runtime -> Change runtime type -> T4 GPU.")

# Check available memory
!free -h
!nvidia-smi

In [ ]:
%%bash
# Install Circom compiler and Node.js toolchain
apt-get update -qq && apt-get install -y -qq build-essential curl git

# Install Circom 2 binary
curl -L https://github.com/iden3/circom/releases/latest/download/circom-linux-amd64 -o /usr/local/bin/circom
chmod +x /usr/local/bin/circom
circom --version

# Install global SnarkJS
npm install -g snarkjs
snarkjs --version

In [ ]:
# Clone repository and install dependencies
!git clone https://github.com/your-username/zk-private-voting.git zk-private-voting || true
%cd zk-private-voting
!npm install

## Section 2: ZK Circuit Compilation and Groth16 Trusted Setup

In [ ]:
%%bash
# 1. Compile Circom circuit to R1CS, WASM, and SYM
mkdir -p circuits/build
circom circuits/voting.circom --r1cs --wasm --sym -o circuits/build --prime bn128

# Print constraint count
snarkjs r1cs info circuits/build/voting.r1cs

In [ ]:
%%bash
# 2. Universal Powers of Tau ceremony
cd circuits/build
snarkjs powersoftau new bn128 10 pot10_0000.ptau -v
echo "entropy" | snarkjs powersoftau contribute pot10_0000.ptau pot10_0001.ptau --name="Contributor"
snarkjs powersoftau prepare phase2 pot10_0001.ptau pot10_final.ptau -v

# 3. Groth16 Circuit Setup
snarkjs groth16 setup voting.r1cs pot10_final.ptau voting_0000.zkey
echo "entropy" | snarkjs zkey contribute voting_0000.zkey voting_final.zkey --name="PragmaContributor"
snarkjs zkey export verificationkey voting_final.zkey verification_key.json

# Export Solidity Verifier contract directly into contracts/
snarkjs zkey export solidityverifier voting_final.zkey ../../contracts/Verifier.sol
echo "Verifier.sol generated successfully."

In [ ]:
# Generate proof from circuits/input.json
!node scripts/generateProof.js

## Section 3: Smart Contract Deployment

In [ ]:
# Run Hardhat unit tests
!npx hardhat test

In [ ]:
# Deploy contracts to Polygon Mumbai / Amoy
import os
# os.environ["PRIVATE_KEY"] = "your_wallet_private_key_here"
# os.environ["POLYGONSCAN_API_KEY"] = "your_polygonscan_api_key_here"
# !npx hardhat run scripts/deploy.js --network mumbai

## Section 4: Frontend Web Testing via Tunneling (ngrok)

In [ ]:
%%bash
cd frontend
npm install
nohup npm run dev -- --host 0.0.0.0 --port 3000 > ../frontend.log 2>&1 &
echo "Frontend server started on port 3000"

In [ ]:
!pip install -q pyngrok
from pyngrok import ngrok
try:
    public_url = ngrok.connect(3000).public_url
    print(f"Public DApp URL: {public_url}")
except Exception as e:
    print(f"Tunneling note: {e}")

## Section 5: Hardware Performance Benchmarking

In [ ]:
# Run 10-iteration empirical benchmark
!node scripts/benchmark.js